# Module 9: Performance & Cost Engineering for AI Systems

Build a caching decorator and profiling workflow to cut latency and spend.


## 1. Setup

Load performance/cost core utilities.


In [1]:
import sys
from pathlib import Path

module_dir = Path.cwd()
if str(module_dir) not in sys.path:
    sys.path.append(str(module_dir))

from perf_cost_core import (
    SemanticCache,
    LLMServiceSimulator,
    caching_decorator,
    batch_infer,
    generate_query_load,
    profile_prompt_optimization,
)

print("✓ Setup complete")


✓ Setup complete


## 2. Baseline Query Cost and Latency


In [2]:
service = LLMServiceSimulator()
baseline = service.answer("What is the daily per diem?")
baseline


2026-03-01 19:05:58 - perf_cost_core - INFO - LLMServiceSimulator initialized
2026-03-01 19:05:58 - perf_cost_core - INFO - Processing LLM request for query: 'What is the daily per diem?...'
2026-03-01 19:05:59 - perf_cost_core - INFO - LLM request completed - Cost: $0.0228, Tokens: 228, Latency: 199.8ms


{'answer': 'The standard daily per diem is $75 for domestic travel.',
 'tokens': 228,
 'latency_ms': 199.8,
 'estimated_cost_usd': Decimal('0.0228')}

## 3. Semantic Caching Decorator

Repeated/near-duplicate queries should return instantly with zero API cost.


In [3]:
cache = SemanticCache(similarity_threshold=0.82)
cached_call = caching_decorator(cache, service)

queries = [
    "What is the daily per diem?",
    "what is daily per diem",
    "What is the meal limit?",
    "what is the meal limit please",
    "What is the daily per diem",
]

run_metrics = []
for q in queries:
    ans, m = cached_call(q)
    run_metrics.append(m.to_dict())

run_metrics


2026-03-01 19:05:59 - perf_cost_core - INFO - SemanticCache initialized with threshold: 0.82
2026-03-01 19:05:59 - perf_cost_core - INFO - Processing LLM request for query: 'What is the daily per diem?...'
2026-03-01 19:05:59 - perf_cost_core - INFO - LLM request completed - Cost: $0.0228, Tokens: 228, Latency: 199.8ms
2026-03-01 19:05:59 - perf_cost_core - INFO - Cache PUT request for query: 'What is the daily per diem?'
2026-03-01 19:05:59 - perf_cost_core - INFO - Cache HIT for query: 'what is daily per diem' -> key: 'What is the daily per diem?', score: 0.8333
2026-03-01 19:05:59 - perf_cost_core - INFO - Processing LLM request for query: 'What is the meal limit?...'
2026-03-01 19:05:59 - perf_cost_core - INFO - LLM request completed - Cost: $0.0228, Tokens: 228, Latency: 199.8ms
2026-03-01 19:05:59 - perf_cost_core - INFO - Cache PUT request for query: 'What is the meal limit?'
2026-03-01 19:05:59 - perf_cost_core - INFO - Cache HIT for query: 'what is the meal limit please' -> ke

[{'query': 'What is the daily per diem?',
  'cached': False,
  'latency_ms': 219.058,
  'estimated_cost_usd': '0.0228',
  'tokens': 228},
 {'query': 'what is daily per diem',
  'cached': True,
  'latency_ms': 7.557,
  'estimated_cost_usd': '0',
  'tokens': 0},
 {'query': 'What is the meal limit?',
  'cached': False,
  'latency_ms': 208.837,
  'estimated_cost_usd': '0.0228',
  'tokens': 228},
 {'query': 'what is the meal limit please',
  'cached': True,
  'latency_ms': 3.457,
  'estimated_cost_usd': '0',
  'tokens': 0},
 {'query': 'What is the daily per diem',
  'cached': True,
  'latency_ms': 4.223,
  'estimated_cost_usd': '0',
  'tokens': 0}]

## 4. Load Test with Repeated Traffic

Simulate many users asking repetitive policy questions.


In [4]:
load_queries = generate_query_load(n=120, seed=42)

hits = 0
misses = 0
total_cost = 0.0
for q in load_queries:
    _, m = cached_call(q)
    if m.cached:
        hits += 1
    else:
        misses += 1
    total_cost += float(m.estimated_cost_usd)

{
    "queries": len(load_queries),
    "hits": hits,
    "misses": misses,
    "cache_hit_rate": round(hits / len(load_queries), 4),
    "total_cost_usd": round(total_cost, 4),
}


2026-03-01 19:05:59 - perf_cost_core - INFO - Cache HIT for query: 'What is the daily per diem please' -> key: 'What is the daily per diem?', score: 0.8571
2026-03-01 19:05:59 - perf_cost_core - INFO - Cache HIT for query: 'What is the meal limit please' -> key: 'What is the meal limit?', score: 0.8333
2026-03-01 19:05:59 - perf_cost_core - INFO - Processing LLM request for query: 'Is coffee reimbursable...'
2026-03-01 19:05:59 - perf_cost_core - INFO - LLM request completed - Cost: $0.0228, Tokens: 228, Latency: 199.8ms
2026-03-01 19:05:59 - perf_cost_core - INFO - Cache PUT request for query: 'Is coffee reimbursable'
2026-03-01 19:05:59 - perf_cost_core - INFO - Cache HIT for query: 'What is the daily per diem please' -> key: 'What is the daily per diem?', score: 0.8571
2026-03-01 19:05:59 - perf_cost_core - INFO - Processing LLM request for query: 'Is coffee reimbursable please...'
2026-03-01 19:05:59 - perf_cost_core - INFO - LLM request completed - Cost: $0.0228, Tokens: 228, Late

{'queries': 120,
 'hits': 116,
 'misses': 4,
 'cache_hit_rate': 0.9667,
 'total_cost_usd': 0.0915}

## 5. Batch Optimization

Compare single-call processing vs batch processing.


In [5]:
batch_stats = batch_infer(load_queries[:60], batch_size=10)
batch_stats


2026-03-01 19:06:00 - perf_cost_core - INFO - LLMServiceSimulator initialized
2026-03-01 19:06:00 - perf_cost_core - INFO - Processing LLM request for query: 'What is the daily per diem please...'
2026-03-01 19:06:00 - perf_cost_core - INFO - LLM request completed - Cost: $0.0228, Tokens: 228, Latency: 199.8ms
2026-03-01 19:06:00 - perf_cost_core - INFO - Processing LLM request for query: 'What is the meal limit please...'
2026-03-01 19:06:01 - perf_cost_core - INFO - LLM request completed - Cost: $0.0228, Tokens: 228, Latency: 199.8ms
2026-03-01 19:06:01 - perf_cost_core - INFO - Processing LLM request for query: 'Is coffee reimbursable...'
2026-03-01 19:06:01 - perf_cost_core - INFO - LLM request completed - Cost: $0.0228, Tokens: 228, Latency: 199.8ms
2026-03-01 19:06:01 - perf_cost_core - INFO - Processing LLM request for query: 'What is the daily per diem please...'
2026-03-01 19:06:01 - perf_cost_core - INFO - LLM request completed - Cost: $0.0228, Tokens: 228, Latency: 199.8ms
2

{'queries': 60,
 'batch_size': 10,
 'single_ms': 12363.78,
 'batch_ms': 3027.26,
 'speedup_x': 4.08}

## 6. Prompt Token Optimization

Shorter system prompts reduce token and cost footprint.


In [6]:
prompt_stats = profile_prompt_optimization("What is the daily per diem?")
prompt_stats


2026-03-01 19:06:16 - perf_cost_core - INFO - LLMServiceSimulator initialized
2026-03-01 19:06:16 - perf_cost_core - INFO - Processing LLM request for query: 'What is the daily per diem?...'
2026-03-01 19:06:16 - perf_cost_core - INFO - LLM request completed - Cost: $0.0308, Tokens: 308, Latency: 227.8ms
2026-03-01 19:06:16 - perf_cost_core - INFO - Processing LLM request for query: 'What is the daily per diem?...'
2026-03-01 19:06:16 - perf_cost_core - INFO - LLM request completed - Cost: $0.0148, Tokens: 148, Latency: 171.8ms


{'long_prompt_tokens': 308,
 'short_prompt_tokens': 148,
 'token_reduction': 160,
 'long_prompt_cost': '0.0308',
 'short_prompt_cost': '0.0148',
 'cost_reduction': '0.0160'}

## 7. Assertions (Smoke Checks)


In [7]:
assert hits > misses
assert batch_stats["speedup_x"] > 1.0
assert int(prompt_stats["token_reduction"]) > 0

print("✓ Module 9 performance/cost smoke checks passed")


✓ Module 9 performance/cost smoke checks passed
